In [ ]:
# ========================
# ES5 — LLM e Prompting
# ========================

!pip install -q torch transformers accelerate

In [ ]:
import transformers
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# caricamento modello (instruction tuned)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print("Caricamento del modello LLM (può richiedere qualche minuto)...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)

model.generation_config.max_length = None

# creazione della pipeline di generazione
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    return_full_text=False
)

Caricamento del modello LLM (può richiedere qualche minuto)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
transformers.logging.set_verbosity_error()

In [ ]:
# 5.1 LABELING DEI TOPIC

# 1. Definizione dei topic da etichettare (da es4)
topics_data = {
    0: "profit, sales, fullquote, percent, stores, earnings",
    1: "sox, red, yankees, series, baseball, league",
    2: "champions, england, league, arsenal, test, cup",
    3: "microsoft, linux, ibm, software, intel, source",
    4: "scientists, species, human, climate, study",
    5: "coach, ranked, state, football, notre, dame",
    6: "nfl, patriots, jets, season, sunday, game",
    7: "space, nasa, moon, mars, spacecraft, station",
    8: "olympic, athens, gold, medal, olympics",
    9: "bush, kerry, president, john, campaign"
}

# 2. Creazione prompt e generazione
def get_topic_label(keywords):
    # formato chat-template richiesto dai modelli Instruct
    messages = [
        {"role": "user",
         "content": f"""
            I will provide you with a list of keywords associated with a specific topic from a news dataset.
            Your task is to provide a single, concise Category Label that best describes the topic.
            Keywords: {keywords}
            Category Label:"""
          }

    ]

    output = pipe(messages, max_length=32)
    return output[0]['generated_text'].strip()

# 3. Esecuzione sui topic
for topic_id, keywords in topics_data.items():
    label = get_topic_label(keywords)
    print(f"Topic {topic_id}: [{keywords}] \n--> LABEL GENERATA: {label}\n")

Topic 0: [profit, sales, fullquote, percent, stores, earnings] 
--> LABEL GENERATA: Financial News

Topic 1: [sox, red, yankees, series, baseball, league] 
--> LABEL GENERATA: Baseball

Topic 2: [champions, england, league, arsenal, test, cup] 
--> LABEL GENERATA: Football (Soccer) League

Topic 3: [microsoft, linux, ibm, software, intel, source] 
--> LABEL GENERATA: Operating Systems and Software

Topic 4: [scientists, species, human, climate, study] 
--> LABEL GENERATA: Category Label: Environmental Science

Topic 5: [coach, ranked, state, football, notre, dame] 
--> LABEL GENERATA: Football

Topic 6: [nfl, patriots, jets, season, sunday, game] 
--> LABEL GENERATA: NFL Game

Topic 7: [space, nasa, moon, mars, spacecraft, station] 
--> LABEL GENERATA: Space Exploration

Topic 8: [olympic, athens, gold, medal, olympics] 
--> LABEL GENERATA: Olympics

Topic 9: [bush, kerry, president, john, campaign] 
--> LABEL GENERATA: U.S. Politics



In [ ]:
# 5.2 IDENTIFICAZIONE TERMINI DATE LE DEFINIZIONI + STRATEGIE DI PROMPTING

file_path = "dataset_definizioni_TLN_25.xlsx"
try:
    dataset_definizioni = pd.read_excel(file_path, sheet_name='Foglio1')
except Exception as e:
    print(f"Errore critico nella lettura: {e}")

# 1. Preparazione dataset
# identifica le colonne delle definizioni
def_cols = [col for col in dataset_definizioni.columns if col.startswith('P') and col[1:].isdigit()]

# crea una riga per ogni coppia termine-definizione (wide->long)
df_long = dataset_definizioni[['Termine'] + def_cols] \
                                .melt(id_vars=['Termine'],
                                      value_vars=def_cols,
                                      var_name='Definizione_ID', # P1, P2, ...
                                      value_name='Definizione') # testo definizione

# rimuove le righe dove la definizione è mancante
df_test_set = df_long.dropna(subset=['Termine', 'Definizione'])

print(f"\nCaricati {len(dataset_definizioni)} termini unici.")
print(f"Eseguo il test su {len(df_test_set)} coppie termine-definizione.")
print("-" * 50)

# 2. Normalizzazione e decoding
def first_word(text: str) -> str:
    """Prende solo la prima parola (token) dal testo generato."""
    if text is None:
        return ""
    text = text.strip()
    if not text:
        return ""
    # prima riga
    text = text.splitlines()[0].strip()
    # se ci sono prefissi tipo "Termine:" o "Parola:"
    text = re.sub(r"^(termine|parola|lemma)\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    # prima parola (separata da spazi)
    return text.split()[0].strip() if text else ""

def normalize_term(term: str) -> str:
    """Normalizza: lowercase, rimuove punteggiatura e gestisce pantaloni->pantalone."""
    term = first_word(term).lower()

    # rimuove caratteri non alfabetici (tieni lettere italiane)
    term = re.sub(r"[^a-zàèéìòù]", "", term)

    # fix morfologico specifico richiesto
    if term == "pantaloni":
        term = "pantalone"

    if term == "euristiche":
        term = "euristica"

    return term

# 3. Esecuzione test e report
def run_guessing_test(test_df, guessing_function, title):
    results = [] # lista di dizionari
    target_words = [] # gold label
    predicted_words = [] # predizioni
    total_tests = len(test_df) # numero totale di righe da testare

    print(f"\n==================================================")
    print(f"Inizio test: {title}...")
    print(f"Eseguo {total_tests} test. Attendere...")

    # loop sui test
    for index, row in test_df.iterrows():
        target_word = row['Termine']
        definition = row['Definizione']

        # esecuzione del guessing
        guessed_word_raw = guessing_function(definition)

        # normalizzazione
        target_clean = normalize_term(target_word)
        guessed_clean = normalize_term(guessed_word_raw)

        target_words.append(target_clean)
        predicted_words.append(guessed_clean)

        # confronto: se coincidono, corretto
        is_correct = "✅ CORRETTO" if target_clean == guessed_clean else "❌ ERRATO"

        results.append({
            "Termine Atteso": target_word,
            "Definizione Usata": definition,
            "Parola Predetta (raw)": guessed_word_raw,
            "Termine Atteso (norm)": target_clean,
            "Parola Predetta (norm)": guessed_clean,
            "Esito": is_correct
        })

        # stampa l'avanzamento
        if index < 16: # stampa solo i primi 10 per brevità nell'output
            print(f"[{index+1}/{total_tests}] Termine: {target_word:<15} | Predetto: {guessed_word_raw:<20} | Norm: {guessed_clean:<12} | Esito: {is_correct}")

    # report finale (numero di predizioni corrette e accuracy)
    correct_guesses = sum(1 for target, pred in zip(target_words, predicted_words) if target == pred)
    accuracy = (correct_guesses / total_tests) * 100 if total_tests > 0 else 0

    print("\n--------------------------------------------------")
    print(f"RISULTATI FINALI {title}:")
    print(f"Indovinate Correttamente: {correct_guesses} / {total_tests}")
    print(f"Accuratezza: {accuracy:.2f}%")
    print("==================================================")

    return pd.DataFrame(results), accuracy

# 4. Strategie di prompting

# 4.1 ZERO-SHOT (Prompt conciso e "specifico")
def guess_word_from_definition_V1_ZeroShot(definition):
    prompt = f"""
Sei un classificatore lessicale in italiano.
Dato una definizione, restituisci il termine corretto.

REGOLE DI OUTPUT:
- Scrivi SOLO il termine, in ITALIANO.
- UNA sola e singola parola, singolare, senza spazi né punteggiatura.
- Non usare inglese.
- Usa la forma più comune e canonica (lemma).

Definizione: {definition}
Termine:"""

    messages = [{"role": "user", "content": prompt}]
    out = pipe(messages, return_full_text=False, use_cache=False, max_new_tokens=8, do_sample=False)
    raw = out[0]["generated_text"]
    return first_word(raw)

# 4.2 FEW-SHOT + CONTESTO SEMANTICO
def guess_word_from_definition_V2_OneShot_SemanticHints(definition):
    prompt = f"""
Sei un classificatore lessicale in italiano.
Dato una definizione, restituisci il termine corretto.

REGOLE DI OUTPUT:
- Scrivi SOLO il termine, in ITALIANO.
- UNA sola e singola parola, singolare, senza spazi né punteggiatura.
- Non usare inglese.
- Usa la forma più comune e canonica (lemma).
- Se predici il termine 'Pantaloni', usa invece il singolare 'Pantalone'.
- Se la definizione descrive una regola pratica per risolvere problemi/decidere rapidamente -> usa il lemma standard.

Esempi:
Definizione: Strumento ottico che ingrandisce oggetti molto piccoli.
Termine: Microscopio
Definizione: Possibilità di subire un danno o una minaccia.
Termine: Pericolo
Definizione: Strategia pratica non garantita ottima per risolvere un problema.
Termine: Euristica
Definizione: Indumento che copre la parte inferiore del corpo e le gambe.
Termine: Pantalone

Definizione: {definition}
Termine:"""

    messages = [{"role": "user", "content": prompt}]
    out = pipe(messages, return_full_text=False, use_cache=False, max_new_tokens=8, do_sample=False)
    raw = out[0]["generated_text"]
    return first_word(raw)

# 5. Esecuzione test

test_subset = df_test_set#.head(16)
print(f"Esecuzione dei test su un sottoinsieme di {len(test_subset)} definizioni per dimostrazione.")

# 1) Zero-Shot specifico/conciso
results_v1, acc_v1 = run_guessing_test(
    test_subset,
    guess_word_from_definition_V1_ZeroShot,
    "V1 - Zero-Shot (prompt conciso + 'più specifica')"
)

# 2) One-Shot + suggerimenti semantici
results_v2, acc_v2 = run_guessing_test(
    test_subset,
    guess_word_from_definition_V2_OneShot_SemanticHints,
    "V2 - Few-Shot + Contesto semantico"
)


Caricati 4 termini unici.
Eseguo il test su 153 coppie termine-definizione.
--------------------------------------------------
Esecuzione dei test su un sottoinsieme di 153 definizioni per dimostrazione.

Inizio test: V1 - Zero-Shot (prompt conciso + 'più specifica')...
Eseguo 153 test. Attendere...
[1/153] Termine: Pantalone       | Predetto: Piedi                | Norm: piedi        | Esito: ❌ ERRATO
[2/153] Termine: Microscopio     | Predetto: Microscopio          | Norm: microscopio  | Esito: ✅ CORRETTO
[3/153] Termine: Pericolo        | Predetto: pericolo             | Norm: pericolo     | Esito: ✅ CORRETTO
[4/153] Termine: Euristica       | Predetto: Ricerca              | Norm: ricerca      | Esito: ❌ ERRATO
[5/153] Termine: Pantalone       | Predetto: Pantaloni            | Norm: pantalone    | Esito: ✅ CORRETTO
[6/153] Termine: Microscopio     | Predetto: Microscopio          | Norm: microscopio  | Esito: ✅ CORRETTO
[7/153] Termine: Pericolo        | Predetto: pericolo       